# 05 - Fine-Tuning on Distorted Data (Part 4)

**Detection + segmentation only.** ORB feature matching and classical optical
flow are non-learned and have no weights to fine-tune - per `CLAUDE.md`'s
Evaluation section ("where applicable" matters), they stop at
clean -> distorted -> restored (notebooks 02-04).

Training labels for the distorted images are reused from clean GT (content
is unchanged by distortion, only pixels are), per the PDF's Part 4 "create
labels from clean". Detection *continues* fine-tuning the clean-adapted
YOLO checkpoint from 02; segmentation gets its first fine-tuning step here
(its 02 baseline was already off-the-shelf, no adaptation needed).

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in str(get_ipython())
REPO_URL = "https://github.com/Shir-Siman-Tov/Image-Processing-Project.git"
REPO_DIR = "/content/Image-Processing-Project"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL} {REPO_DIR}")
    else:
        # Runtime already had this repo cloned from an earlier cell run in this
        # session - pull so we don't keep running against a stale checkout.
        get_ipython().system(f"git -C {REPO_DIR} pull")
    os.chdir(REPO_DIR)
    get_ipython().system("pip install -q -e .")
    get_ipython().system("pip install -q -r requirements.txt")

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from ultralytics import YOLO

from ipproj import config
from ipproj.datasets import kitti, kitti_flow, kitti_yolo_format
from ipproj.datasets.kitti import read_image
from ipproj.datasets.materialize import materialize_transformed
from ipproj.tasks import object_detection, semantic_segmentation
from ipproj.distortions import REGISTRY as DISTORTIONS
from ipproj.viz.plotting import plot_metric_vs_intensity, save_figure

detection_splits = kitti.load_object_detection_subset()
segmentation_splits = kitti.load_semantic_segmentation_subset()

checkpoint_path = (config.CHECKPOINT_ROOT / "yolo_clean_baseline_path.txt").read_text().strip()

distortion_results = pd.read_csv(config.RESULTS_ROOT / "03_distortions.csv")
restoration_df = pd.read_csv(config.RESULTS_ROOT / "04_restoration.csv")
clean_baseline = pd.read_csv(config.RESULTS_ROOT / "02_clean_baseline.csv").set_index("metric")["value"]

## Fine-tune per distortion (strongest intensity, mirroring the PDF's Part 4 example)

In [ ]:
fine_tuned_results = []

for name, module in DISTORTIONS.items():
    level = len(module.LEVELS) - 1
    distort_fn = lambda img, m=module, l=level: m.distort(img, l)

    # --- detection: continue fine-tuning the clean-adapted checkpoint ---
    distorted_train_detection = materialize_transformed(
        detection_splits["train"], distort_fn, config.DISTORTED_ROOT / name / "train_detection"
    )
    distorted_data_yaml = kitti_yolo_format.build_yolo_dataset(
        {"train": distorted_train_detection, "val": detection_splits["val"], "test": detection_splits["test"]},
        output_dir=config.KITTI_ROOT / "yolo_format_distorted" / name,
    )
    continued_model = YOLO(checkpoint_path)
    train_results = continued_model.train(
        data=str(distorted_data_yaml),
        epochs=config.YOLO_FINE_TUNE_EPOCHS,
        seed=config.RANDOM_SEED,
        project=str(config.CHECKPOINT_ROOT / "yolo"),
        name=f"finetuned_{name}",
    )
    finetuned_yolo = YOLO(Path(train_results.save_dir) / "weights" / "best.pt")

    training_curve = object_detection.load_training_curve(Path(train_results.save_dir) / "weights" / "best.pt")
    fig = plot_metric_vs_intensity(
        training_curve["epoch"],
        {"train loss": training_curve["train_loss"], "val loss": training_curve["val_loss"]},
        xlabel="epoch", ylabel="loss", title=f"{name}: YOLOv8 fine-tuning train vs val loss",
    )
    save_figure(fig, f"05_finetuning_distorted/yolo_training_curve_{name}.png")

    distorted_test_detection = materialize_transformed(
        detection_splits["test"], distort_fn, config.DISTORTED_ROOT / name / "test_detection"
    )
    finetuned_detection_metrics = object_detection.evaluate(finetuned_yolo, distorted_test_detection)

    # --- segmentation: first fine-tuning step ---
    distorted_train_segmentation = materialize_transformed(
        segmentation_splits["train"], distort_fn, config.DISTORTED_ROOT / name / "train_segmentation"
    )
    finetuned_segformer, finetuned_processor = semantic_segmentation.load_pretrained()
    semantic_segmentation.fine_tune(finetuned_segformer, finetuned_processor, distorted_train_segmentation)

    distorted_test_segmentation = materialize_transformed(
        segmentation_splits["test"], distort_fn, config.DISTORTED_ROOT / name / "test_segmentation"
    )
    finetuned_segmentation_metrics = semantic_segmentation.evaluate(
        finetuned_segformer, finetuned_processor, distorted_test_segmentation
    )

    fine_tuned_results.append({
        "distortion": name,
        "level": level,
        "map_finetuned": float(finetuned_detection_metrics["map"]),
        "map_50_finetuned": float(finetuned_detection_metrics["map_50"]),
        "map_75_finetuned": float(finetuned_detection_metrics["map_75"]),
        "mar_100_finetuned": float(finetuned_detection_metrics["mar_100"]),
        "mean_iou_finetuned": float(finetuned_segmentation_metrics.mean()),
    })

fine_tuned_df = pd.DataFrame(fine_tuned_results)
config.RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
fine_tuned_df.to_csv(config.RESULTS_ROOT / "05_finetuning.csv", index=False)
fine_tuned_df

## Final comparison: clean -> distorted -> restored -> fine-tuned

In [ ]:
max_level_distorted = distortion_results.loc[distortion_results.groupby("distortion")["level"].idxmax()]
max_level_restored = restoration_df.loc[restoration_df.groupby("distortion")["level"].idxmax()]

final_comparison = fine_tuned_df.merge(
    max_level_distorted[["distortion", "level", "map", "map_50", "mean_iou"]].rename(
        columns={"map": "map_distorted", "map_50": "map_50_distorted", "mean_iou": "mean_iou_distorted"}
    ),
    on=["distortion", "level"],
).merge(
    max_level_restored[["distortion", "level", "map", "map_50", "mean_iou"]].rename(
        columns={"map": "map_restored", "map_50": "map_50_restored", "mean_iou": "mean_iou_restored"}
    ),
    on=["distortion", "level"],
)
final_comparison["map_clean"] = clean_baseline["map"]
final_comparison["map_50_clean"] = clean_baseline["map_50"]
final_comparison["mean_iou_clean"] = clean_baseline["mean_iou"]
final_comparison = final_comparison[[
    "distortion",
    "map_clean", "map_distorted", "map_restored", "map_finetuned",
    "map_50_clean", "map_50_distorted", "map_50_restored", "map_50_finetuned",
    "mean_iou_clean", "mean_iou_distorted", "mean_iou_restored", "mean_iou_finetuned",
]]
final_comparison

## Per-distortion final comparison plots

In [ ]:
stages = ["clean", "distorted", "restored", "fine-tuned"]

for _, row in final_comparison.iterrows():
    fig = plot_metric_vs_intensity(
        stages,
        {
            "mAP": [row["map_clean"], row["map_distorted"], row["map_restored"], row["map_finetuned"]],
            "mAP@0.5": [row["map_50_clean"], row["map_50_distorted"], row["map_50_restored"], row["map_50_finetuned"]],
            "mean IoU": [row["mean_iou_clean"], row["mean_iou_distorted"], row["mean_iou_restored"], row["mean_iou_finetuned"]],
        },
        xlabel="pipeline stage", ylabel="metric value",
        title=f"{row['distortion']}: clean -> distorted -> restored -> fine-tuned",
    )
    save_figure(fig, f"05_finetuning_distorted/final_comparison_{row['distortion']}.png")